# Watch two trained models play

Loads a white and a black checkpoint, plays one game, then lets you scrub through it.
For each position you see the board, the move played (SAN), the mover's value output,
the material balance from the mover's side, and **value + material**: under
potential-based shaping the value head learns `V - material`, so add material back
before reading whether the model thinks it is winning (see `learning/records/0003`).

In [1]:
import torch
from src.model.chess_model import ChessPolicyProbs
from src.viz.play import play_game

# Pick a checkpoint pair. Available locally:
#   experiments/potential-based-shaping/          scale 1.0 baseline
#   experiments/potential-based-shaping/scale02/  scale 0.2 (current default)
import glob
CKPT_DIR = "experiments/potential-based-shaping/scale02"
WHITE_CKPT = sorted(glob.glob(f"{CKPT_DIR}/white_model_*.pth"))[-1]
BLACK_CKPT = sorted(glob.glob(f"{CKPT_DIR}/black_model_*.pth"))[-1]
print("white:", WHITE_CKPT)
print("black:", BLACK_CKPT)

def load_model(path):
    model = ChessPolicyProbs()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    return model.eval()

white_model = load_model(WHITE_CKPT)
black_model = load_model(BLACK_CKPT)

white: experiments/potential-based-shaping/scale02/white_model_20260915161133_episodes_500.pth
black: experiments/potential-based-shaping/scale02/black_model_20260915161133_episodes_500.pth


In [ ]:
# greedy=True plays the most likely move every time; False samples from the policy.
# SEED fixes the game so you can re-inspect it; change it (or use None) for a different game.
SEED = 0
game = play_game(white_model, black_model, greedy=True, max_moves=300, seed=SEED)
print(f"{len(game.moves)} moves, result: {game.result}")

91 moves, result: white_win


In [3]:
import html, json, uuid
import chess, chess.svg
from IPython.display import display, HTML

def render_replay(game, size=400):
    frames = []
    board = chess.Board()
    frames.append((chess.svg.board(board, size=size), "<b>Start position</b>"))
    for rec in game.moves:
        board = chess.Board(rec.fen_after)
        top = ", ".join(f"{s} {p:.2f}" for s, p in rec.top_moves)
        info = (f"<b>Move {rec.move_number}</b> {rec.side} plays <b>{html.escape(rec.san)}</b> "
                f"(p={rec.prob_played:.2f})<br>"
                f"value = {rec.value:+.2f} &nbsp; material = {rec.material:+.0f} &nbsp; "
                f"<b>value + material = {rec.value_plus_material:+.2f}</b><br>top-3: {html.escape(top)}")
        last = chess.Board(rec.fen_before).parse_san(rec.san)
        frames.append((chess.svg.board(board, size=size, lastmove=last), info))
    n = len(frames) - 1
    uid = uuid.uuid4().hex[:8]           # unique per render: reruns must not collide
    payload = json.dumps([[s, i] for s, i in frames])
    return f"""
<div id="rp_{uid}">
  <input type="range" min="0" max="{n}" value="0" style="width:600px">
  <span class="ply">ply 0 / {n}</span> &nbsp; result: <b>{game.result}</b> &nbsp; ({n} plies, render {uid})
  <div style="display:flex;gap:24px;align-items:flex-start;margin-top:8px">
    <div class="board"></div><div class="info" style="font-family:monospace"></div>
  </div>
</div>
<script>
(function () {{
  const root = document.getElementById("rp_{uid}");
  const frames = {payload};
  const show = (i) => {{
    root.querySelector(".board").innerHTML = frames[i][0];
    root.querySelector(".info").innerHTML = frames[i][1];
    root.querySelector(".ply").textContent = "ply " + i + " / {n}";
  }};
  root.querySelector("input").addEventListener("input", (e) => show(e.target.value));
  show(0);
}})();
</script>"""

display(HTML(render_replay(game)))

In [ ]:
# Value vs value+material over the game, per side.
import matplotlib.pyplot as plt

for side, color in (("white", "tab:blue"), ("black", "tab:red")):
    recs = [m for m in game.moves if m.side == side]
    x = [m.move_number for m in recs]
    plt.plot(x, [m.value for m in recs], color=color, alpha=0.4, label=f"{side} value (raw)")
    plt.plot(x, [m.value_plus_material for m in recs], color=color, label=f"{side} value + material")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("ply"); plt.ylabel("mover's perspective"); plt.legend(); plt.title(f"result: {game.result}")
plt.show()